In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/archive.zip"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/drive-download-20260518T140706Z-3-001.zip"

In [ ]:
import os

print(os.listdir('/content'))

In [ ]:
import json

train_data = []

with open('/content/balanced_train.json', 'r') as f:

    for line in f:

        train_data.append(json.loads(line))

print(len(train_data))

In [ ]:
import os
import json
import torch
import matplotlib.pyplot as plt

from PIL import Image

from tqdm import tqdm

from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
train_data = []

with open('/content/balanced_train.json', 'r') as f:

    for line in f:

        train_data.append(json.loads(line))

In [ ]:
import os

print(os.listdir('/content'))

In [ ]:
val_data = []

with open('/content/val_data.json', 'r') as f:

    for line in f:

        val_data.append(json.loads(line))

In [ ]:
labels = sorted(
    list(set(item['class_label'] for item in train_data))
)

label_to_idx = {
    label: idx for idx, label in enumerate(labels)
}

idx_to_label = {
    idx: label for label, idx in label_to_idx.items()
}

num_classes = len(labels)

print(num_classes)

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor()
])

val_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor()
])

In [ ]:
class IndoFashionDataset(Dataset):

    def __init__(
        self,
        data_list,
        label_to_idx,
        transform=None
    ):

        self.data_list = data_list

        self.label_to_idx = label_to_idx

        self.transform = transform

    def __len__(self):

        return len(self.data_list)

    def __getitem__(self, idx):

        item = self.data_list[idx]

        image_path = "/content/" + item['image_path']

        image = Image.open(image_path).convert("RGB")

        label = self.label_to_idx[item['class_label']]

        if self.transform:

            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = IndoFashionDataset(
    train_data,
    label_to_idx,
    train_transform
)

val_dataset = IndoFashionDataset(
    val_data,
    label_to_idx,
    val_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
model = models.mobilenet_v3_large(pretrained=True)

num_features = model.classifier[3].in_features

model.classifier[3] = nn.Linear(
    num_features,
    num_classes
)

model = model.to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=2,
    factor=0.5
)

In [ ]:
scaler = torch.cuda.amp.GradScaler()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
train_accuracies = []

val_accuracies = []

train_losses = []

val_losses = []

best_val_accuracy = 0

epochs = 10

for epoch in range(epochs):

    model.train()

    train_loss = 0.0

    train_correct = 0

    train_total = 0

    train_loop = tqdm(train_loader)

    for images, labels in train_loop:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        _, predicted = torch.max(
            outputs,
            1
        )

        train_total += labels.size(0)

        train_correct += (
            predicted == labels
        ).sum().item()

        train_loop.set_description(
            f"Epoch {epoch+1}"
        )

        train_loop.set_postfix(

            loss=loss.item(),

            accuracy=
            100 * train_correct / train_total
        )

    train_accuracy = (

        100 * train_correct / train_total
    )

    avg_train_loss = (

        train_loss / len(train_loader)
    )

    model.eval()

    val_loss = 0.0

    val_correct = 0

    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            _, predicted = torch.max(
                outputs,
                1
            )

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()

    val_accuracy = (

        100 * val_correct / val_total
    )

    avg_val_loss = (

        val_loss / len(val_loader)
    )

    train_accuracies.append(
        train_accuracy
    )

    val_accuracies.append(
        val_accuracy
    )

    train_losses.append(
        avg_train_loss
    )

    val_losses.append(
        avg_val_loss
    )

    print(f"\nEpoch {epoch+1} Completed")

    print(
        f"Train Loss: {avg_train_loss:.4f}"
    )

    print(
        f"Train Accuracy: {train_accuracy:.2f}%"
    )

    print(
        f"Validation Loss: {avg_val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: {val_accuracy:.2f}%"
    )

    torch.save(

        model.state_dict(),

        f"/content/drive/MyDrive/mobilenet_epoch_{epoch+1}.pth"
    )

    print(
        f"Checkpoint Saved For Epoch {epoch+1}"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(

            model.state_dict(),

            "/content/drive/MyDrive/best_mobilenet_model.pth"
        )

        print("Best model saved!")

    scheduler.step(val_accuracy)

In [ ]:
import os

files = os.listdir('/content/drive/MyDrive')

for file in files:

    if "mobilenet" in file:

        print(file)

In [ ]:
train_accuracies = [
    78.04,
    87.46,
    89.82,
    90.97,
    92.14,
    93.21,
    94.16,
    94.66,
    96.07,
    96.59
]

val_accuracies = [
    85.57,
    87.12,
    87.89,
    88.03,
    88.07,
    87.81,
    87.93,
    87.05,
    87.63,
    87.60
]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    train_accuracies,
    label='Train Accuracy'
)

plt.plot(
    val_accuracies,
    label='Validation Accuracy'
)

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.title("MobileNet Accuracy Graph")

plt.legend()

plt.savefig(
    "/content/drive/MyDrive/mobilenet_accuracy_graph.png"
)

plt.show()

In [ ]:
train_losses = [
    0.7071,
    0.3826,
    0.3087,
    0.2589,
    0.2215,
    0.1879,
    0.1631,
    0.1443,
    0.1040,
    0.0884
]

val_losses = [
    0.4351,
    0.3932,
    0.3755,
    0.3675,
    0.3781,
    0.4049,
    0.4417,
    0.4478,
    0.4710,
    0.4985
]

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    label='Train Loss'
)

plt.plot(
    val_losses,
    label='Validation Loss'
)

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("MobileNet Loss Graph")

plt.legend()

plt.savefig(
    "/content/drive/MyDrive/mobilenet_loss_graph.png"
)

plt.show()

In [ ]:
test_data = []

with open('/content/test_data.json', 'r') as f:

    for line in f:

        test_data.append(json.loads(line))

In [ ]:
test_dataset = IndoFashionDataset(
    test_data,
    label_to_idx,
    val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/best_mobilenet_model.pth",
        map_location=device
    )
)

model.eval()

print("Best MobileNet Model Loaded")

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

In [ ]:
test_accuracy = accuracy_score(
    all_labels,
    all_preds
)

print(
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)

In [ ]:
precision = precision_score(
    all_labels,
    all_preds,
    average='weighted'
)

print(
    f"Precision: {precision:.4f}"
)

In [ ]:
recall = recall_score(
    all_labels,
    all_preds,
    average='weighted'
)

print(
    f"Recall: {recall:.4f}"
)

In [ ]:
f1 = f1_score(
    all_labels,
    all_preds,
    average='weighted'
)

print(
    f"F1 Score: {f1:.4f}"
)

In [ ]:
from sklearn.metrics import classification_report

import pandas as pd

import matplotlib.pyplot as plt

report = classification_report(
    all_labels,
    all_preds,
    output_dict=True
)

report_df = pd.DataFrame(
    report
).transpose()

plt.figure(figsize=(12,8))

plt.axis('off')

table = plt.table(

    cellText=report_df.round(2).values,

    colLabels=report_df.columns,

    rowLabels=report_df.index,

    loc='center'
)

table.auto_set_font_size(False)

table.set_fontsize(10)

table.scale(1.2, 1.2)

plt.title(
    "MobileNet Classification Report",
    pad=20
)

plt.savefig(
    "/content/drive/MyDrive/mobilenet_classification_report.png",
    bbox_inches='tight'
)

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

import seaborn as sns

import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted Labels")

plt.ylabel("True Labels")

plt.title("MobileNet Confusion Matrix")

plt.savefig(
    "/content/drive/MyDrive/mobilenet_confusion_matrix.png"
)

plt.show()

In [ ]:
print("MobileNet Project Saved Successfully")